<a href="https://colab.research.google.com/github/Harshavardhan655/complier-design-/blob/main/cd_week_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

!apt-get update -qq
!apt-get install -y flex gcc


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:13.2.0-7ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison flex-doc
The following NEW packages will be installed:
  flex libfl-dev libfl2
0 upgraded, 3 newly installed, 0 to remove and 59 not upgraded.
Need to get 317 kB of archives.
After this operation, 1,068 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/main amd64 flex amd64 2.6.4-8.2build1 [301 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libfl2 amd64 2.6.4-8.2build1 [9,768 B]
Get:3 http://archive.ubuntu.com/ubuntu noble/main amd64 libfl-dev amd64 2.6.4-8.2build1 [6,250 B

In [2]:

%%writefile expression.l
%{
#include <stdio.h>

int paren = 0, square = 0, question = 0;
int bad = 0;
%}

%%
"(" {
    paren++;
    printf("TOKEN: (\n");
}

")" {
    paren--;
    printf("TOKEN: )\n");
    if (paren < 0) bad = 1;
}

"[" {
    square++;
    printf("TOKEN: [\n");
}

"]" {
    square--;
    printf("TOKEN: ]\n");
    if (square < 0) bad = 1;
}

"?" {
    question++;
    printf("TOKEN: ?\n");
}

[ \t\r\n]+   ;

. {
    /* Ignore other characters */
}
%%

int main(void)
{
    printf("Enter expression (Ctrl+D to finish):\n");
    yylex();

    printf("\n");

    if (paren == 0 && !bad)
        printf("Parentheses are Balanced\n");
    else
        printf("Parentheses are Unbalanced\n");

    if (square == 0 && !bad)
        printf("Square Brackets are Balanced\n");
    else
        printf("Square Brackets are Unbalanced\n");

    printf("Total '?' operators found: %d\n", question);

    return 0;
}

int yywrap(void)
{
    return 1;
}


Writing expression.l


In [3]:

!flex expression.l
!gcc lex.yy.c -o expression -lfl
!echo 'a = (b + c) * arr[5]; x = (a > b) ? a : b;' | ./expression


Enter expression (Ctrl+D to finish):
TOKEN: (
TOKEN: )
TOKEN: [
TOKEN: ]
TOKEN: (
TOKEN: )
TOKEN: ?

Parentheses are Balanced
Square Brackets are Balanced
Total '?' operators found: 1


In [4]:

%%writefile ipv4.l
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int valid_ipv4(const char *ip)
{
    int parts = 0;
    const char *p = ip;

    while (*p) {
        if (*p < '0' || *p > '9')
            return 0;

        int value = 0;
        int digits = 0;

        while (*p >= '0' && *p <= '9') {
            value = value * 10 + (*p - '0');
            digits++;
            if (value > 255)
                return 0;
            p++;
        }

        if (digits == 0 || digits > 3)
            return 0;

        parts++;

        if (*p == '.') {
            p++;
            if (*p == '\0')
                return 0;
        } else if (*p != '\0') {
            return 0;
        }
    }

    return parts == 4;
}
%}

%%
[^\n]+ {
    char ip[1024];
    snprintf(ip, sizeof(ip), "%s", yytext);

    /* Remove trailing carriage return, if present */
    ip[strcspn(ip, "\r")] = '\0';

    if (valid_ipv4(ip))
        printf("Valid IPv4 Address: %s\n", ip);
    else
        printf("Invalid IPv4 Address: %s\n", ip);
}

\n   ;
%%

int main(void)
{
    printf("Enter IPv4 addresses, one per line:\n");
    yylex();
    return 0;
}

int yywrap(void)
{
    return 1;
}


Writing ipv4.l


In [5]:

!flex ipv4.l
!gcc lex.yy.c -o ipv4 -lfl

!printf '192.168.1.1\n10.0.0.255\n255.255.255.0\n256.100.50.25\n192.168.1\n192.168.1.abc\n' | ./ipv4


Enter IPv4 addresses, one per line:
Valid IPv4 Address: 192.168.1.1
Valid IPv4 Address: 10.0.0.255
Valid IPv4 Address: 255.255.255.0
Invalid IPv4 Address: 256.100.50.25
Invalid IPv4 Address: 192.168.1
Invalid IPv4 Address: 192.168.1.abc
